## ML model interpretability

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier

### setup backtesting:
* #### load features from feature store 
* #### load best model and hyparams from registry

In [ ]:
# load features from feature store
PATH_FEATURE_STORE = "datasets/feature_store/v1/"
df_train = pd.read_csv(PATH_FEATURE_STORE + "train_data.csv")
df_test = pd.read_csv(PATH_FEATURE_STORE + "test_data.csv")

# load the trained model from model registry
import joblib
PATH_MODEL_REGISTRY = "model_registry/v1/"
best_model = joblib.load(PATH_MODEL_REGISTRY + "best_model.pkl")

# load the best hyperparameters from model registry
best_hyperparams = joblib.load(PATH_MODEL_REGISTRY + "best_hyperparams.pkl")
print("Best Hyperparameters:", best_hyperparams)

### Global Interpretability


* #### impurity-based feature importance (FI)

* #### permutation-based FI

* #### permutation test p-value FI

### Impurity FI

In [ ]:
# plot feature importance
feature_importance = best_model.feature_importances_
feature_names = df_train.drop(columns=['label_idx']).columns
feature_importance_df = pd.DataFrame({
                                    'feature': feature_names,
                                    'importance': feature_importance
                                })

FileNotFoundError: [Errno 2] No such file or directory: 'model_registry/v1/best_model.pkl'

In [ ]:
df_plot = feature_importance_df.sort_values(by='importance', ascending=False)

In [ ]:


plt.figure(figsize=(10, 10))
sns.barplot(x='importance', y='feature', data=df_plot, palette='Blues', hue='feature')
plt.title('Impurity based Feature Importance (Gradient Boosting)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### Permutation FI

In [ ]:
from sklearn.inspection import permutation_importance

NUM_REPEATS = 5
RANDOM_STATE = 0

X_train = df_train.drop(columns=['label_idx'])
y_train = df_train['label_idx']

importance_results = permutation_importance(best_model, 
                                            X_train, 
                                            y_train, 
                                            n_repeats=NUM_REPEATS,
                                            random_state=RANDOM_STATE
                                            )

feature_names = df_train.drop(columns=['label_idx']).columns
feature_importance_df = pd.DataFrame({
                            'feature': feature_names,
                            'importance_mean': importance_results.importances_mean,
                            'importance_std': importance_results.importances_std
                        })

In [ ]:
import matplotlib.pyplot as plt

df_plot = feature_importance_df.sort_values("importance_mean", ascending=False)

plt.figure(figsize=(10, 10))
ax = sns.barplot(data=df_plot, x="importance_mean", y="feature", palette="Blues", orient="h", hue="feature")
ax.errorbar(
            x=df_plot["importance_mean"],
            y=range(len(df_plot)),
            xerr=df_plot["importance_std"],
            fmt="none",
            ecolor="gray",
            capsize=2
            )
ax.set(title="Permutation-based Feature Importance (Gradient Boosting)", xlabel="Mean Permutation Importance", ylabel="Feature")

plt.show()

### Permutation Test p-value FI

In [ ]:
NUM_PERMUTATIONS = 2

list_permutation_importances = []
for i in range(NUM_PERMUTATIONS):
    y_train_permuted = np.random.permutation(y_train)
    copy_model = GradientBoostingClassifier(**hyp_param_search.best_params_, random_state=42)
    copy_model.fit(X_train, y_train_permuted)
    list_permutation_importances.append(copy_model.feature_importances_)

p_values = []
for i in range(len(feature_names)):
    original_importance = feature_importance[i]
    permuted_importances = [perm[i] for perm in list_permutation_importances]
    p_value = np.mean([1 if perm >= original_importance else 0 for perm in permuted_importances])
    p_values.append(p_value)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Build DataFrame
feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importance,
    "p_value": p_values
})

alpha = 0.05  # significance threshold

# Sort once and reset index for consistent plotting
df_plot = feature_importance_df.sort_values("importance", ascending=False).reset_index(drop=True)

# Flag significance
df_plot["significant"] = df_plot["p_value"] < alpha

# Plot
plt.figure(figsize=(10, 10))

# Base bars
ax = sns.barplot(
    data=df_plot,
    x="importance",
    y="feature",
    color="blue",
    orient="h",
    alpha=0.3
)

# Overlay significant ones
sns.barplot(
    data=df_plot[df_plot["significant"]],
    x="importance",
    y="feature",
    color="red",
    orient="h",
    ax=ax,
    alpha=0.7
)

# Annotate p-values
for i, row in df_plot.iterrows():
    ax.text(
        row["importance"] + 0.001,
        i,
        f"p-value={row['p_value']:.2e}",
        va="center",
        fontsize=9
    )

# Labels & title
ax.set_title("Permutation Test Feature Importance (Gradient Boosting)")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")

plt.tight_layout()
plt.show()

### Local Intepretability